<a href="https://colab.research.google.com/github/anapedretti/dio_assistente_voz_gemini/blob/main/Assistente_de_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
language = 'pt'
top_level_domain = 'com.br'

# Gravação de áudio com Python e JS

In [5]:
from IPython.display import Audio, display, Javascript, HTML
from google.colab import output
from base64 import b64decode

# HTML e CSS: Criamos um botão e o container da barra (inicialmente escondido)
display(HTML("""
<style>
    #audio-interface {
        padding: 20px;
        background: #f9f9f9;
        border-radius: 10px;
        border: 1px solid #ddd;
        text-align: center;
        width: 300px;
    }
    #start-btn {
        background-color: #4CAF50;
        color: white;
        padding: 10px 20px;
        border: none;
        border-radius: 5px;
        cursor: pointer;
        font-weight: bold;
    }
    #start-btn:hover { background-color: #45a049; }

    #progress-container {
        width: 100%;
        background-color: #e0e0e0;
        border-radius: 5px;
        display: none; /* Escondido até o clique */
        margin-top: 10px;
    }
    #progress-bar {
        width: 0%;
        height: 20px;
        background-color: #2196F3;
        border-radius: 5px;
        transition: width 0.1s;
    }
</style>

<div id="audio-interface">
    <button id="start-btn">🎤 Iniciar Gravação</button>
    <div id="progress-container">
        <div id="progress-bar"></div>
        <small id="progress-text">Pronto</small>
    </div>
</div>
"""))

# JavaScript: A classe aguarda o evento de clique e ativa a gravação, mostrando a barra de progresso até acabar o tempo estabelecido
RECORD_JS = """
class ManualRecorder {
    constructor(duration) {
        this.duration = duration;
        this.btn = document.getElementById('start-btn');
        this.container = document.getElementById('progress-container');
        this.bar = document.getElementById('progress-bar');
        this.text = document.getElementById('progress-text');
    }

    async start() {
        // Retorna uma Promise que resolve apenas quando a gravação termina
        return new Promise(resolve => {
            this.btn.onclick = async () => {
                // Ajusta a interface
                this.btn.style.display = 'none';
                this.container.style.display = 'block';

                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                const recorder = new MediaRecorder(stream);
                const chunks = [];

                recorder.ondataavailable = e => chunks.push(e.data);

                recorder.onstop = async () => {
                    const blob = new Blob(chunks);
                    const reader = new FileReader();
                    reader.readAsDataURL(blob);
                    reader.onloadend = () => {
                        stream.getTracks().forEach(t => t.stop());
                        this.container.style.display = 'none';
                        this.btn.style.display = 'inline-block'; // Reseta o botão
                        resolve(reader.result);
                    };
                };

                recorder.start();
                this.updateBar();

                setTimeout(() => recorder.stop(), this.duration);
            };
        });
    }

    updateBar() {
        let start = Date.now();
        let interval = setInterval(() => {
            let elapsed = Date.now() - start;
            let percent = Math.min((elapsed / this.duration) * 100, 100);
            this.bar.style.width = percent + '%';
            this.text.innerText = `Gravando... ${Math.round(percent)}%`;
            if (elapsed >= this.duration) clearInterval(interval);
        }, 100);
    }
}

var runManualRecord = async (time) => {
    const rec = new ManualRecorder(time);
    return await rec.start();
}
"""
# Mude o valor de 'sec' para alterar o tempo em que a gravação ficará ativa
def record_manual(sec=5):
    display(Javascript(RECORD_JS))
    print("Aguardando clique no botão para iniciar...")

    try:
        # O Python fica parado aqui até o usuário clicar e a gravação terminar
        js_result = output.eval_js(f'runManualRecord({sec * 1000})')

        audio_data = b64decode(js_result.split(',')[1])
        file_name = 'audio_input.wav'
        with open(file_name, 'wb') as f:
            f.write(audio_data)

        print("Áudio capturado!")
        return file_name
    except Exception as e:
        print(f"Erro: {e}")
        return None

# Execução
arquivo = record_manual(5)
if arquivo:
    display(Audio(arquivo))

<IPython.core.display.Javascript object>

Aguardando clique no botão para iniciar...
Áudio capturado!


# Reconhecimento de fala e resposta à pergunta com a Gemini API

In [6]:
from google.colab import userdata
from google import genai
import os

# Substitua o texto "CHAVE API AQUI" por sua API Key do Google AI Studio
client = genai.Client(api_key='CHAVE API AQUI')

myfile = client.files.upload(file=arquivo)

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=myfile
)

print(response.text)

Com certeza! Imagina que a luz do sol parece branca, mas, na verdade, ela é como uma **caixa de giz de cera** escondida. Dentro dessa luz branca, existem todas as cores do arco-íris: vermelho, laranja, amarelo, verde, azul, anil e violeta.

Aqui está o que acontece:

1.  **A "Viagem" da Luz:** A luz do sol viaja pelo espaço até chegar na Terra. Quando ela chega aqui, ela encontra a nossa **atmosfera**, que é como uma camada de ar cheia de gases e partículas minúsculas que a gente não consegue ver.
2.  **Obstáculos no Caminho:** Essas partículas de ar funcionam como pequenos obstáculos. Quando a luz bate nelas, ela se espalha para todos os lados.
3.  **O "Pulo" das Cores:** Cada cor viaja de um jeito. As cores como o vermelho e o amarelo viajam em ondas longas e calmas, então elas conseguem passar pelos obstáculos quase sem bater em nada.
4.  **A Agitação do Azul:** Já a cor **azul** viaja em ondas curtas e muito rápidas. Por ser mais "agitada", ela bate nas partículas de ar e sai pulan

# Sintetizando a resposta do Gemini como voz (gTTS)

In [12]:
!pip install gTTS
from gtts import gTTS
import re

def limpar_texto_para_voz(texto):
    # Remove os asteriscos duplos (negrito) ou simples (itálico)
    texto_limpo = re.sub(r'\*+', '', texto)

    # Opcional: Remove hashtags (títulos) se houver
    texto_limpo = re.sub(r'#+', '', texto_limpo)

    # Opcional: Remove links markdown [texto](link) mantendo apenas o texto
    texto_limpo = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', texto_limpo)

    return texto_limpo

texto_para_gtts = limpar_texto_para_voz(response.text)

print(texto_para_gtts)

# Cria um objeto gTTS com a resposta gerada pelo Gemini e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=texto_para_gtts, lang=language, tld = top_level_domain, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=False))